# **TASK 1: ADVANCED WEB SCRAPING — Real Estate Data**

**Step 1: Install & Import Libraries**

In [1]:
# Installing the tools we need
!pip install requests beautifulsoup4 pandas

# Importing them
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

print("All good! Libraries are ready.")

All good! Libraries are ready.


**Step 2: Scrape the Data**

In [7]:
# Added Ahmedabad to the list
cities = ["mumbai", "delhi", "bangalore", "ahmedabad"]

all_properties = []

for city in cities:
    print(f"Scraping {city}...")

    url = f"https://www.magicbricks.com/property-for-sale/residential-real-estate?cityName={city}&page=1"
    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, "lxml")

    cards = soup.find_all("div", class_="mb-srp__card")
    print(f"  Found {len(cards)} listings")

    for card in cards:
        title_tag = card.find("h2", class_="mb-srp__card--title")
        title = title_tag.text.strip() if title_tag else "N/A"
        location = title_tag["title"] if title_tag and title_tag.has_attr("title") else "N/A"

        price_tag = card.find("div", class_="mb-srp__card__price--amount")
        price = price_tag.text.strip() if price_tag else "N/A"

        area_tag = card.find("div", {"data-summary": "carpet-area"})
        area = area_tag.find("div", class_="mb-srp__card__summary--value").text.strip() if area_tag else "N/A"

        status_tag = card.find("div", {"data-summary": "status"})
        status = status_tag.find("div", class_="mb-srp__card__summary--value").text.strip() if status_tag else "N/A"

        all_properties.append({
            "City": city.capitalize(),
            "Title": title,
            "Location": location,
            "Price": price,
            "Area": area,
            "Status": status
        })

    time.sleep(2)

df = pd.DataFrame(all_properties)
print(f"\nTotal properties: {len(df)}")
print(df["City"].value_counts())  # Shows count per city

Scraping mumbai...
  Found 30 listings
Scraping delhi...
  Found 0 listings
Scraping bangalore...
  Found 30 listings
Scraping ahmedabad...
  Found 30 listings

Total properties: 90
City
Mumbai       30
Bangalore    30
Ahmedabad    30
Name: count, dtype: int64


**Step 3: Preview the Data**

In [8]:
# Convert our list into a table
df = pd.DataFrame(all_properties)

# See the first 10 rows
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
df.head(10)

Total rows: 90
Total columns: 6


,City,Title,Location,Price,Area,Status
0,Mumbai,"3 BHK Flat for Sale in RNA NG Eclat, RNA NG ...","3 BHK Flat for Sale in RNA NG Eclat, RNA NG ...",₹4.60 Cr,1116 sqft,Ready to Move
1,Mumbai,"4 BHK Flat for Sale in Worli, Mumbai","4 BHK Flat for Sale in Worli, Mumbai",₹30.03 Cr,3562 sqft,Poss. by Dec '30
2,Mumbai,3 BHK Flat for Sale in Krishna Residency Atm...,3 BHK Flat for Sale in Krishna Residency Atm...,₹3.80 Cr,N/A,Ready to Move
3,Mumbai,"2 BHK Flat for Sale in Gujarat Kasturi Van, ...","2 BHK Flat for Sale in Gujarat Kasturi Van, ...",₹1.14 Cr,764 sqft,Poss. by Dec '28
4,Mumbai,3 BHK Flat for Sale in Hiranandani Gardens G...,3 BHK Flat for Sale in Hiranandani Gardens G...,₹5.25 Cr,1500 sqft,Ready to Move
5,Mumbai,"3 BHK Flat for Sale in Ruparel Iris, Ruparel...","3 BHK Flat for Sale in Ruparel Iris, Ruparel...",₹9.88 Cr,1356 sqft,N/A
6,Mumbai,"2 BHK Flat for Sale in Temple hill, Temple h...","2 BHK Flat for Sale in Temple hill, Temple h...",₹3 Cr,850 sqft,Ready to Move
7,Mumbai,"3 BHK Flat for Sale in Chandak Sarvam, Chand...","3 BHK Flat for Sale in Chandak Sarvam, Chand...",₹4.11 Cr,1100 sqft,Poss. by Dec '30
8,Mumbai,"3 BHK Flat for Sale in Ruparel Stardom, Rupa...","3 BHK Flat for Sale in Ruparel Stardom, Rupa...",₹3.57 Cr,1260 sqft,Poss. by Dec '28
9,Mumbai,"3 BHK Flat for Sale in Lodha Bellevue, Lodha...","3 BHK Flat for Sale in Lodha Bellevue, Lodha...",₹5.97 Cr,1162 sqft,Poss. by Dec '26


**Step 4 (Fix): Find the Right Location Tag**

In [9]:
# Let's peek inside one card's HTML to find the correct location tag
response = requests.get(
    "https://www.magicbricks.com/property-for-sale/residential-real-estate?cityName=mumbai&page=1",
    headers=headers
)
soup = BeautifulSoup(response.text, "lxml")

# Grab just the first card
first_card = soup.find("div", class_="mb-srp__card")

# Print its full HTML so we can see what's inside
print(first_card)

<div class="mb-srp__card preferred-agent"><div class="mb-srp__card__container"><div class="mb-srp__card__photo"><div class="mb-srp__card__photo__fig"><span class="mb-srp__card__photo__fig--count">8<span class="sign-plus">+</span> Photos</span><img alt="Buy 3 BHK Resale Flat in  RNA NG Eclat Mumbai" class="mb-srp__card__photo__fig--graphic zoom" decoding="async" fetchpriority="high" src="https://img.staticmb.com/mbphoto/property/cropped_images/ver2/XIwvQlc61t8ZIpanz4mAnTVG88A9ds7UYqNUnvS7WWcjmsgr_qjG/Photo_h300_w450/70218381_12_PropertyImage1-9217619501910033_300_450.jpg" title="Buy 3 BHK Resale Flat in  RNA NG Eclat Mumbai"/><span class="icon-video"></span><div class="mb-srp__card__photo__fig--post">Posted:  <!-- --> <!-- -->Yesterday</div></div><div class="mb-srp__card__ads"><div class="mb-srp__card__ads__info"><div class="mb-srp__card__ads__info--letter">A</div><div class="mb-srp__card__ads__info--right"><div class="mb-srp__card__ads__info--name">BMG Properties</div><div class="mb-sr

**Step 5: Fix & Re-Scrape with Correct Tags**

In [10]:
all_properties = []

for city in cities:
    print(f"Scraping {city}...")

    url = f"https://www.magicbricks.com/property-for-sale/residential-real-estate?cityName={city}&page=1"
    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, "lxml")

    cards = soup.find_all("div", class_="mb-srp__card")
    print(f"  Found {len(cards)} listings")

    for card in cards:
        # Title
        title_tag = card.find("h2", class_="mb-srp__card--title")
        title = title_tag.text.strip() if title_tag else "N/A"

        # Location is inside the title attribute of h2
        location = title_tag["title"] if title_tag and title_tag.has_attr("title") else "N/A"

        # Price
        price_tag = card.find("div", class_="mb-srp__card__price--amount")
        price = price_tag.text.strip() if price_tag else "N/A"

        # Area
        area_tag = card.find("div", {"data-summary": "carpet-area"})
        area = area_tag.find("div", class_="mb-srp__card__summary--value").text.strip() if area_tag else "N/A"

        # Status
        status_tag = card.find("div", {"data-summary": "status"})
        status = status_tag.find("div", class_="mb-srp__card__summary--value").text.strip() if status_tag else "N/A"

        all_properties.append({
            "City": city.capitalize(),
            "Title": title,
            "Location": location,
            "Price": price,
            "Area": area,
            "Status": status
        })

    time.sleep(2)

# Preview
df = pd.DataFrame(all_properties)
print(f"\nTotal properties: {len(df)}")
df.head(5)

Scraping mumbai...
  Found 30 listings
Scraping delhi...
  Found 0 listings
Scraping bangalore...
  Found 30 listings
Scraping ahmedabad...
  Found 30 listings

Total properties: 90


,City,Title,Location,Price,Area,Status
0,Mumbai,"3 BHK Flat for Sale in RNA NG Eclat, RNA NG ...","3 BHK Flat for Sale in RNA NG Eclat, RNA NG ...",₹4.60 Cr,1116 sqft,Ready to Move
1,Mumbai,"4 BHK Flat for Sale in Worli, Mumbai","4 BHK Flat for Sale in Worli, Mumbai",₹30.03 Cr,3562 sqft,Poss. by Dec '30
2,Mumbai,3 BHK Flat for Sale in Krishna Residency Atm...,3 BHK Flat for Sale in Krishna Residency Atm...,₹3.80 Cr,N/A,Ready to Move
3,Mumbai,"2 BHK Flat for Sale in Gujarat Kasturi Van, ...","2 BHK Flat for Sale in Gujarat Kasturi Van, ...",₹1.14 Cr,764 sqft,Poss. by Dec '28
4,Mumbai,3 BHK Flat for Sale in Hiranandani Gardens G...,3 BHK Flat for Sale in Hiranandani Gardens G...,₹5.25 Cr,1500 sqft,Ready to Move


**Step 6: Save the Data as a CSV File**

In [13]:
# Save to a CSV file
df.to_csv("real_estate_data.csv", index=False)

print(f"Saved! Total properties: {len(df)}")
print(f"Columns: {list(df.columns)}")

# Download it to your computer
from google.colab import files
files.download("real_estate_data.csv")

Saved! Total properties: 90
Columns: ['City', 'Title', 'Location', 'Price', 'Area', 'Status']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>